# ESM Function-Prediction Probes

This notebook adds prediction-oriented probes to the ESM GO embedding explorer.
It uses the same six balanced organism/domain datasets produced by `esm_go_embedding_analysis.ipynb` and asks three follow-up questions:

1. **ESM kNN transfer:** if a protein inherits the GO group of its nearest ESM neighbors, how often is the direct GO group recovered?
2. **Sequence-identity stratification:** does the ESM kNN signal remain after excluding close sequence neighbors by BLASTP coverage-adjusted identity?
3. **Linear probe:** can a simple linear classifier recover the selected GO group from frozen ESM embeddings?

For E. coli, when S2F prediction scores are available locally, the notebook also compares the selected GO-term ranking against the ESM probes. Human S2F score comparison is marked unavailable unless matching S2F prediction files exist locally.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'plm.py').exists():
    raise FileNotFoundError(f'Run from the S2F repository or notebooks directory: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))
import plm

DATA_DIR = PROJECT_ROOT / 'notebooks' / 'esm_go_explorer' / 'data'
INDEX_PATH = DATA_DIR / 'index.json'
PROBE_RESULTS_PATH = DATA_DIR / 'prediction_probe_results.json'
KNN_SUMMARY_PATH = DATA_DIR / 'knn_transfer_summary.csv'
LINEAR_SUMMARY_PATH = DATA_DIR / 'linear_probe_summary.csv'
S2F_COMPARISON_PATH = DATA_DIR / 's2f_probe_comparison.csv'
S2F_SELECTED_SCORES_PATH = DATA_DIR / 's2f_selected_scores_ecoli.csv'

ECOLI_CACHE = Path('/media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings')
HUMAN_CACHE = Path('/media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot')
CACHE_BY_ORGANISM = {
    'ecoli': ECOLI_CACHE,
    'human': HUMAN_CACHE,
}
S2F_PREDICTION_BY_ORGANISM = {
    'ecoli': Path('/media/marcelo_baez/HD_Disc1/.S2F/output/83333_w_plm_heavy/prediction.df'),
    'human': None,
}

BLASTP = Path('/usr/local/ncbi-blast-2.17.0+/bin/blastp')
MAKEBLASTDB = Path('/usr/local/ncbi-blast-2.17.0+/bin/makeblastdb')
RANDOM_SEED = 42
K_VALUES = [1, 3, 5, 10]
IDENTITY_FILTERS = [
    {'key': 'all', 'label': 'All neighbors', 'max_identity': None},
    {'key': 'lt50', 'label': '<50% sequence identity', 'max_identity': 50.0},
    {'key': 'lt40', 'label': '<40% sequence identity', 'max_identity': 40.0},
    {'key': 'lt30', 'label': '<30% sequence identity', 'max_identity': 30.0},
]

def load_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

index_payload = load_json(INDEX_PATH)
datasets = {}
for descriptor in index_payload['datasets']:
    payload = load_json(PROJECT_ROOT / 'notebooks' / 'esm_go_explorer' / descriptor['path'])
    datasets[descriptor['dataset_id']] = payload

print(f'Loaded {len(datasets)} frontend datasets from {DATA_DIR}')
print('Datasets:', ', '.join(datasets))


Loaded 6 frontend datasets from /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data
Datasets: ecoli_mf, ecoli_bp, ecoli_cc, human_mf, human_bp, human_cc


## Load Embeddings and Sequences

The probe uses the same frozen ESM-1b embeddings from `plm.py`. Sequences are loaded from the FASTA paths recorded in each embedding cache metadata, only for the selected proteins.


In [2]:
def cache_index(cache):
    return {protein_id: index for index, protein_id in enumerate(cache.ids)}

caches = {}
cache_indices = {}
for organism_key, cache_path in CACHE_BY_ORGANISM.items():
    cache = plm.load_embedding_cache(cache_path, validate=False)
    if cache is None:
        raise RuntimeError(f'Could not load embedding cache for {organism_key}: {cache_path}')
    caches[organism_key] = cache
    cache_indices[organism_key] = cache_index(cache)
    print(organism_key, cache.embeddings.shape, cache_path)

selected_by_organism = defaultdict(set)
for dataset in datasets.values():
    organism_key = dataset['metadata']['organism_key']
    selected_by_organism[organism_key].update(protein['protein_id'] for protein in dataset['proteins'])


def read_selected_sequences_from_fastas(fasta_paths, selected_ids, protein_id_mode='uniprot'):
    selected_ids = set(selected_ids)
    found = {}
    remaining = set(selected_ids)
    for fasta_path in fasta_paths:
        fasta_path = Path(fasta_path)
        if not fasta_path.exists():
            raise FileNotFoundError(fasta_path)
        header = None
        chunks = []
        with plm.open_maybe_gzip(fasta_path) as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line:
                    continue
                if line.startswith('>'):
                    if header is not None:
                        protein_id = plm.protein_id_from_header(header, protein_id_mode)
                        if protein_id in remaining:
                            found[protein_id] = plm.NON_AA_RE.sub('', ''.join(chunks).upper())
                            remaining.remove(protein_id)
                            if not remaining:
                                return found
                    header = line[1:]
                    chunks = []
                else:
                    chunks.append(line)
            if header is not None:
                protein_id = plm.protein_id_from_header(header, protein_id_mode)
                if protein_id in remaining:
                    found[protein_id] = plm.NON_AA_RE.sub('', ''.join(chunks).upper())
                    remaining.remove(protein_id)
                    if not remaining:
                        return found
    if remaining:
        raise RuntimeError(f'Missing {len(remaining)} selected sequence(s): {sorted(remaining)[:10]}')
    return found

sequences_by_organism = {}
for organism_key, selected_ids in selected_by_organism.items():
    cache = caches[organism_key]
    fasta_paths = [entry['path'] for entry in cache.metadata['fasta']]
    sequences_by_organism[organism_key] = read_selected_sequences_from_fastas(
        fasta_paths,
        selected_ids,
        protein_id_mode=cache.metadata.get('protein_id_mode', 'uniprot'),
    )
    print(organism_key, len(sequences_by_organism[organism_key]), 'selected sequences loaded')


ecoli (16075, 1280) /media/marcelo_baez/HD_Disc1/.S2F/output/83333/83333_PLM/query_embeddings
human (20420, 1280) /media/marcelo_baez/HD_Disc1/.S2F/data/PLM/embeddings/human_9606_all_swissprot
ecoli 177 selected sequences loaded
human 179 selected sequences loaded


## Sequence Identity by BLASTP

For each 60-protein dataset, BLASTP is run all-vs-all. The identity used for filtering is coverage-adjusted:

`coverage_adjusted_identity = pident * alignment_length / max(query_length, subject_length)`

This avoids treating a short conserved fragment as full-protein high identity.


In [3]:
def write_dataset_fasta(dataset_id, proteins, sequences, out_dir):
    fasta_path = out_dir / f'{dataset_id}.fasta'
    with fasta_path.open('w', encoding='utf-8') as handle:
        for protein in proteins:
            protein_id = protein['protein_id']
            handle.write(f'>{protein_id}\n')
            seq = sequences[protein_id]
            for start in range(0, len(seq), 80):
                handle.write(seq[start:start + 80] + '\n')
    return fasta_path


def blast_identity_matrix(dataset_id, proteins, sequences):
    n = len(proteins)
    ids = [protein['protein_id'] for protein in proteins]
    if not BLASTP.exists() or not MAKEBLASTDB.exists():
        raise RuntimeError('BLASTP/makeblastdb not found; cannot compute sequence identity stratification.')
    identity = np.zeros((n, n), dtype=np.float32)
    np.fill_diagonal(identity, 100.0)
    with tempfile.TemporaryDirectory(prefix=f's2f_{dataset_id}_blast_') as tmp:
        tmp_dir = Path(tmp)
        fasta_path = write_dataset_fasta(dataset_id, proteins, sequences, tmp_dir)
        db_prefix = tmp_dir / dataset_id
        out_path = tmp_dir / f'{dataset_id}.blast.tsv'
        subprocess.run(
            [str(MAKEBLASTDB), '-in', str(fasta_path), '-dbtype', 'prot', '-out', str(db_prefix)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        subprocess.run(
            [
                str(BLASTP),
                '-query', str(fasta_path),
                '-db', str(db_prefix),
                '-out', str(out_path),
                '-outfmt', '6 qseqid sseqid pident length qlen slen evalue bitscore',
                '-evalue', '1000000',
                '-max_target_seqs', str(n),
                '-seg', 'no',
                '-num_threads', '2',
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        id_to_index = {protein_id: index for index, protein_id in enumerate(ids)}
        with out_path.open('r', encoding='utf-8') as handle:
            for line in handle:
                qseqid, sseqid, pident, length, qlen, slen, *_ = line.rstrip('\n').split('\t')
                if qseqid not in id_to_index or sseqid not in id_to_index:
                    continue
                q = id_to_index[qseqid]
                s = id_to_index[sseqid]
                pident = float(pident)
                length = float(length)
                qlen = float(qlen)
                slen = float(slen)
                coverage_adjusted = pident * length / max(qlen, slen)
                if coverage_adjusted > identity[q, s]:
                    identity[q, s] = coverage_adjusted
        identity = np.maximum(identity, identity.T)
        np.fill_diagonal(identity, 100.0)
    return identity

sequence_identity_matrices = {}
for dataset_id, dataset in datasets.items():
    organism_key = dataset['metadata']['organism_key']
    identity = blast_identity_matrix(dataset_id, dataset['proteins'], sequences_by_organism[organism_key])
    sequence_identity_matrices[dataset_id] = identity
    upper = identity[np.triu_indices_from(identity, k=1)]
    print(dataset_id, 'median identity', float(np.median(upper)), 'max non-self identity', float(np.max(upper)))


ecoli_mf median identity 2.291680335998535 max non-self identity 59.77003860473633
ecoli_bp median identity 2.426321506500244 max non-self identity 43.90280532836914
ecoli_cc median identity 1.763127088546753 max non-self identity 20.467836380004883
human_mf median identity 2.179180383682251 max non-self identity 58.3780632019043
human_bp median identity 1.9789705276489258 max non-self identity 39.566131591796875
human_cc median identity 2.123553991317749 max non-self identity 95.79399871826172


## ESM kNN Annotation Transfer

Each protein is held out and assigned the majority GO group among its nearest ESM neighbors. We evaluate `k = 1, 3, 5, 10` under four sequence-identity filters.


In [4]:
def macro_metrics(y_true, y_pred, labels):
    if len(y_true) == 0:
        return {'accuracy': math.nan, 'macro_f1': math.nan, 'balanced_accuracy': math.nan}
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }


def weighted_vote(neighbor_labels, neighbor_scores, label_values):
    votes = defaultdict(float)
    for label, score in zip(neighbor_labels, neighbor_scores):
        votes[int(label)] += max(float(score), 0.0)
    if not votes or sum(votes.values()) <= 0:
        counts = Counter(map(int, neighbor_labels))
        votes = {label: float(count) for label, count in counts.items()}
    return min(label_values, key=lambda label: (-votes.get(int(label), 0.0), int(label)))


def first_same_rank(labels, ranked_indices, query_index):
    query_label = labels[query_index]
    for rank, neighbor_index in enumerate(ranked_indices, start=1):
        if labels[neighbor_index] == query_label:
            return rank
    return None


def run_knn_transfer(dataset_id, dataset):
    proteins = dataset['proteins']
    labels = np.asarray([protein['group_index'] for protein in proteins], dtype=int)
    label_values = sorted(set(labels.tolist()))
    cosine_similarity = np.asarray(dataset['matrices']['cosine_similarity']['matrix'], dtype=np.float32)
    identity = sequence_identity_matrices[dataset_id]
    summary_rows = []
    prediction_rows = []

    for identity_filter in IDENTITY_FILTERS:
        max_identity = identity_filter['max_identity']
        for k in K_VALUES:
            y_true = []
            y_pred = []
            effective_ks = []
            hit_flags = []
            first_same_ranks = []
            for query_index, protein in enumerate(proteins):
                candidates = [index for index in range(len(proteins)) if index != query_index]
                if max_identity is not None:
                    candidates = [index for index in candidates if identity[query_index, index] < max_identity]
                ranked = sorted(candidates, key=lambda index: (-cosine_similarity[query_index, index], index))
                eligible = ranked[:k]
                if not eligible:
                    prediction_rows.append({
                        'dataset_id': dataset_id,
                        'protein_id': protein['protein_id'],
                        'true_group_index': int(labels[query_index]),
                        'true_go_id': protein['go_id'],
                        'identity_filter': identity_filter['key'],
                        'k': k,
                        'effective_k': 0,
                        'predicted_group_index': None,
                        'predicted_go_id': None,
                        'correct': None,
                        'hit_at_k': None,
                        'first_same_rank': None,
                        'top_neighbor': None,
                        'top_neighbor_similarity': None,
                        'top_neighbor_identity': None,
                    })
                    continue
                pred_label = weighted_vote(labels[eligible], cosine_similarity[query_index, eligible], label_values)
                pred_group = dataset['groups'][pred_label]
                same_rank = first_same_rank(labels, ranked, query_index)
                hit_at_k = any(labels[index] == labels[query_index] for index in eligible)
                y_true.append(int(labels[query_index]))
                y_pred.append(int(pred_label))
                effective_ks.append(len(eligible))
                hit_flags.append(bool(hit_at_k))
                first_same_ranks.append(same_rank if same_rank is not None else math.nan)
                top = eligible[0]
                prediction_rows.append({
                    'dataset_id': dataset_id,
                    'protein_id': protein['protein_id'],
                    'true_group_index': int(labels[query_index]),
                    'true_go_id': protein['go_id'],
                    'identity_filter': identity_filter['key'],
                    'k': k,
                    'effective_k': len(eligible),
                    'predicted_group_index': int(pred_label),
                    'predicted_go_id': pred_group['go_id'],
                    'correct': bool(pred_label == labels[query_index]),
                    'hit_at_k': bool(hit_at_k),
                    'first_same_rank': same_rank,
                    'top_neighbor': proteins[top]['protein_id'],
                    'top_neighbor_similarity': float(cosine_similarity[query_index, top]),
                    'top_neighbor_identity': float(identity[query_index, top]),
                })
            metrics = macro_metrics(y_true, y_pred, label_values)
            covered = len(y_true)
            summary_rows.append({
                'dataset_id': dataset_id,
                'organism': dataset['metadata']['organism'],
                'organism_key': dataset['metadata']['organism_key'],
                'domain': dataset['metadata']['domain'],
                'domain_name': dataset['metadata']['domain_name'],
                'identity_filter': identity_filter['key'],
                'identity_filter_label': identity_filter['label'],
                'max_identity': max_identity,
                'k': k,
                'coverage': float(covered / len(proteins)),
                'covered_proteins': int(covered),
                'mean_effective_k': float(np.mean(effective_ks)) if effective_ks else 0.0,
                'hit_at_k': float(np.mean(hit_flags)) if hit_flags else math.nan,
                'mean_first_same_rank': float(np.nanmean(first_same_ranks)) if first_same_ranks else math.nan,
                **metrics,
            })
    return summary_rows, prediction_rows

knn_summary_rows = []
knn_prediction_rows = []
for dataset_id, dataset in datasets.items():
    summary, predictions = run_knn_transfer(dataset_id, dataset)
    knn_summary_rows.extend(summary)
    knn_prediction_rows.extend(predictions)

KNN_SUMMARY_DF = pd.DataFrame(knn_summary_rows)
KNN_PREDICTION_DF = pd.DataFrame(knn_prediction_rows)
display(KNN_SUMMARY_DF.sort_values(['dataset_id', 'identity_filter', 'k']).head(20))


,dataset_id,organism,organism_key,domain,domain_name,identity_filter,identity_filter_label,max_identity,k,coverage,covered_proteins,mean_effective_k,hit_at_k,mean_first_same_rank,accuracy,macro_f1,balanced_accuracy
16,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,all,All neighbors,NaN,1,1.0,60,1.0,0.733333,1.933333,0.733333,0.723366,0.733333
17,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,all,All neighbors,NaN,3,1.0,60,3.0,0.866667,1.933333,0.750000,0.743103,0.750000
18,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,all,All neighbors,NaN,5,1.0,60,5.0,0.900000,1.933333,0.750000,0.739044,0.750000
19,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,all,All neighbors,NaN,10,1.0,60,10.0,0.983333,1.933333,0.683333,0.666656,0.683333
28,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt30,<30% sequence identity,30.0,1,1.0,60,1.0,0.733333,1.933333,0.733333,0.723366,0.733333
29,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt30,<30% sequence identity,30.0,3,1.0,60,3.0,0.866667,1.933333,0.750000,0.743103,0.750000
30,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt30,<30% sequence identity,30.0,5,1.0,60,5.0,0.900000,1.933333,0.750000,0.739044,0.750000
31,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt30,<30% sequence identity,30.0,10,1.0,60,10.0,0.983333,1.933333,0.683333,0.666656,0.683333
24,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt40,<40% sequence identity,40.0,1,1.0,60,1.0,0.733333,1.933333,0.733333,0.723366,0.733333
25,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,lt40,<40% sequence identity,40.0,3,1.0,60,3.0,0.866667,1.933333,0.750000,0.743103,0.750000


## Linear Probe

A simple linear logistic-regression probe is trained on frozen ESM embeddings using leave-one-out cross-validation. This asks whether the selected GO-group labels are linearly accessible from the embeddings.


In [5]:
def dataset_embeddings(dataset):
    organism_key = dataset['metadata']['organism_key']
    cache = caches[organism_key]
    index = cache_indices[organism_key]
    positions = [index[protein['protein_id']] for protein in dataset['proteins']]
    return np.asarray(cache.embeddings[positions], dtype=np.float32)


def run_linear_probe(dataset_id, dataset):
    proteins = dataset['proteins']
    labels = np.asarray([protein['group_index'] for protein in proteins], dtype=int)
    label_values = sorted(set(labels.tolist()))
    X = dataset_embeddings(dataset)
    predictions = []
    probability_rows = []
    y_true = []
    y_pred = []
    true_probabilities = []

    for test_index in range(len(proteins)):
        train_mask = np.ones(len(proteins), dtype=bool)
        train_mask[test_index] = False
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                solver='lbfgs',
                class_weight='balanced',
                max_iter=5000,
                random_state=RANDOM_SEED,
            ),
        )
        model.fit(X[train_mask], labels[train_mask])
        predicted = int(model.predict(X[[test_index]])[0])
        probabilities = model.predict_proba(X[[test_index]])[0]
        class_to_probability = {int(cls): float(prob) for cls, prob in zip(model.classes_, probabilities)}
        true_label = int(labels[test_index])
        true_probability = class_to_probability.get(true_label, 0.0)
        y_true.append(true_label)
        y_pred.append(predicted)
        true_probabilities.append(true_probability)
        predictions.append({
            'dataset_id': dataset_id,
            'protein_id': proteins[test_index]['protein_id'],
            'true_group_index': true_label,
            'true_go_id': proteins[test_index]['go_id'],
            'predicted_group_index': predicted,
            'predicted_go_id': dataset['groups'][predicted]['go_id'],
            'correct': bool(predicted == true_label),
            'true_probability': true_probability,
            'predicted_probability': class_to_probability.get(predicted, 0.0),
        })
        for group in dataset['groups']:
            probability_rows.append({
                'dataset_id': dataset_id,
                'protein_id': proteins[test_index]['protein_id'],
                'go_id': group['go_id'],
                'group_index': int(group['group_index']),
                'probability': class_to_probability.get(int(group['group_index']), 0.0),
            })

    metrics = macro_metrics(y_true, y_pred, label_values)
    cm = confusion_matrix(y_true, y_pred, labels=label_values)
    per_group = []
    for group in dataset['groups']:
        group_index = int(group['group_index'])
        mask = np.asarray(y_true) == group_index
        recall = float(np.mean(np.asarray(y_pred)[mask] == group_index)) if mask.any() else math.nan
        per_group.append({
            'group_index': group_index,
            'go_id': group['go_id'],
            'go_name': group['name'],
            'recall': recall,
            'support': int(mask.sum()),
        })
    summary = {
        'dataset_id': dataset_id,
        'organism': dataset['metadata']['organism'],
        'organism_key': dataset['metadata']['organism_key'],
        'domain': dataset['metadata']['domain'],
        'domain_name': dataset['metadata']['domain_name'],
        'cv': 'leave_one_out',
        'model': 'StandardScaler + LogisticRegression(lbfgs, balanced)',
        'protein_count': len(proteins),
        'class_count': len(label_values),
        'baseline_accuracy': float(1.0 / len(label_values)),
        'mean_true_probability': float(np.mean(true_probabilities)),
        **metrics,
        'confusion_matrix': cm.astype(int).tolist(),
        'per_group': per_group,
    }
    return summary, predictions, probability_rows

linear_summaries = []
linear_predictions = []
linear_probability_rows = []
for dataset_id, dataset in datasets.items():
    summary, predictions, probabilities = run_linear_probe(dataset_id, dataset)
    linear_summaries.append(summary)
    linear_predictions.extend(predictions)
    linear_probability_rows.extend(probabilities)

LINEAR_SUMMARY_DF = pd.DataFrame([{k: v for k, v in row.items() if k not in {'confusion_matrix', 'per_group'}} for row in linear_summaries])
LINEAR_PREDICTION_DF = pd.DataFrame(linear_predictions)
display(LINEAR_SUMMARY_DF)


,dataset_id,organism,organism_key,domain,domain_name,cv,model,protein_count,class_count,baseline_accuracy,mean_true_probability,accuracy,macro_f1,balanced_accuracy
0,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.830172,0.850000,0.849817,0.850000
1,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.750419,0.816667,0.816667,0.816667
2,ecoli_cc,Escherichia coli,ecoli,CC,Cellular Component,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.683335,0.716667,0.713644,0.716667
3,human_mf,Homo sapiens,human,MF,Molecular Function,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.688875,0.733333,0.729461,0.733333
4,human_bp,Homo sapiens,human,BP,Biological Process,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.712192,0.783333,0.785907,0.783333
5,human_cc,Homo sapiens,human,CC,Cellular Component,leave_one_out,"StandardScaler + LogisticRegression(lbfgs, bal...",60,3,0.333333,0.752630,0.800000,0.799885,0.800000


## S2F Score Comparison Where Available

For E. coli, the notebook extracts S2F scores for only the selected proteins and selected GO terms from the local `prediction.df`. Human comparison is reported as unavailable unless an equivalent local prediction file is present.


In [6]:
def extract_s2f_scores(prediction_path, proteins, terms, cache_path):
    proteins = sorted(set(proteins))
    terms = sorted(set(terms))
    expected_pairs = {(protein, term) for protein in proteins for term in terms}
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        cached_pairs = set(zip(cached['protein_id'], cached['go_id']))
        if expected_pairs.issubset(cached_pairs):
            return cached[cached['protein_id'].isin(proteins) & cached['go_id'].isin(terms)].copy()

    rows = []
    found = set()
    proteins_set = set(proteins)
    terms_set = set(terms)
    start_time = time.time()
    with Path(prediction_path).open('r', encoding='utf-8', errors='replace') as handle:
        for line_number, line in enumerate(handle, start=1):
            parts = line.rstrip('\n').split('\t')
            if len(parts) != 3:
                continue
            protein_id, go_id, score_text = parts
            if protein_id in proteins_set and go_id in terms_set:
                pair = (protein_id, go_id)
                if pair not in found:
                    rows.append({
                        'protein_id': protein_id,
                        'go_id': go_id,
                        's2f_score': float(score_text),
                    })
                    found.add(pair)
                    if len(found) == len(expected_pairs):
                        break
            if line_number % 20_000_000 == 0:
                print(f'Scanned {line_number:,} S2F rows; found {len(found):,}/{len(expected_pairs):,} selected scores after {time.time() - start_time:.1f}s')
    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=['protein_id', 'go_id', 's2f_score'])
    # Fill missing selected pairs with zero so every dataset has a complete 3-term ranking.
    existing = set(zip(df['protein_id'], df['go_id']))
    missing_rows = [
        {'protein_id': protein_id, 'go_id': go_id, 's2f_score': 0.0}
        for protein_id, go_id in sorted(expected_pairs - existing)
    ]
    if missing_rows:
        df = pd.concat([df, pd.DataFrame(missing_rows)], ignore_index=True)
    df.to_csv(cache_path, index=False)
    return df

all_ecoli_proteins = []
all_ecoli_terms = []
for dataset in datasets.values():
    if dataset['metadata']['organism_key'] == 'ecoli':
        all_ecoli_proteins.extend(protein['protein_id'] for protein in dataset['proteins'])
        all_ecoli_terms.extend(group['go_id'] for group in dataset['groups'])

s2f_scores_by_organism = {}
ecoli_prediction_path = S2F_PREDICTION_BY_ORGANISM['ecoli']
if ecoli_prediction_path and ecoli_prediction_path.exists():
    print('Extracting/reusing E. coli S2F selected scores from', ecoli_prediction_path)
    s2f_scores_by_organism['ecoli'] = extract_s2f_scores(
        ecoli_prediction_path,
        all_ecoli_proteins,
        all_ecoli_terms,
        S2F_SELECTED_SCORES_PATH,
    )
    print('E. coli S2F selected score rows:', len(s2f_scores_by_organism['ecoli']))
else:
    s2f_scores_by_organism['ecoli'] = pd.DataFrame(columns=['protein_id', 'go_id', 's2f_score'])


def run_s2f_dataset_comparison(dataset_id, dataset):
    organism_key = dataset['metadata']['organism_key']
    prediction_path = S2F_PREDICTION_BY_ORGANISM.get(organism_key)
    if not prediction_path or not Path(prediction_path).exists() or organism_key not in s2f_scores_by_organism:
        return {
            'dataset_id': dataset_id,
            'organism': dataset['metadata']['organism'],
            'organism_key': organism_key,
            'domain': dataset['metadata']['domain'],
            'domain_name': dataset['metadata']['domain_name'],
            'available': False,
            'reason': 'No local S2F prediction.df configured for this organism.',
        }, []

    score_df = s2f_scores_by_organism[organism_key]
    term_to_group = {group['go_id']: int(group['group_index']) for group in dataset['groups']}
    dataset_terms = [group['go_id'] for group in dataset['groups']]
    predictions = []
    y_true = []
    y_pred = []
    for protein in dataset['proteins']:
        protein_scores = score_df[
            (score_df['protein_id'] == protein['protein_id']) & score_df['go_id'].isin(dataset_terms)
        ].copy()
        scores = {term: 0.0 for term in dataset_terms}
        for row in protein_scores.itertuples(index=False):
            scores[row.go_id] = float(row.s2f_score)
        predicted_go = min(dataset_terms, key=lambda term: (-scores[term], term))
        predicted_group = term_to_group[predicted_go]
        true_group = int(protein['group_index'])
        y_true.append(true_group)
        y_pred.append(predicted_group)
        predictions.append({
            'dataset_id': dataset_id,
            'protein_id': protein['protein_id'],
            'true_group_index': true_group,
            'true_go_id': protein['go_id'],
            'predicted_group_index': predicted_group,
            'predicted_go_id': predicted_go,
            'correct': bool(predicted_group == true_group),
            'predicted_score': scores[predicted_go],
            'true_score': scores[protein['go_id']],
            'scores': scores,
        })
    label_values = sorted(term_to_group.values())
    metrics = macro_metrics(y_true, y_pred, label_values)
    summary = {
        'dataset_id': dataset_id,
        'organism': dataset['metadata']['organism'],
        'organism_key': organism_key,
        'domain': dataset['metadata']['domain'],
        'domain_name': dataset['metadata']['domain_name'],
        'available': True,
        'prediction_path': str(prediction_path),
        'protein_count': len(dataset['proteins']),
        **metrics,
    }
    return summary, predictions

s2f_summaries = []
s2f_predictions = []
for dataset_id, dataset in datasets.items():
    summary, predictions = run_s2f_dataset_comparison(dataset_id, dataset)
    s2f_summaries.append(summary)
    s2f_predictions.extend(predictions)

S2F_SUMMARY_DF = pd.DataFrame([{k: v for k, v in row.items() if k != 'reason'} for row in s2f_summaries])
display(S2F_SUMMARY_DF)


Extracting/reusing E. coli S2F selected scores from /media/marcelo_baez/HD_Disc1/.S2F/output/83333_w_plm_heavy/prediction.df
Scanned 20,000,000 S2F rows; found 0/1,593 selected scores after 23.0s
Scanned 40,000,000 S2F rows; found 0/1,593 selected scores after 42.3s
Scanned 60,000,000 S2F rows; found 0/1,593 selected scores after 63.7s
Scanned 80,000,000 S2F rows; found 0/1,593 selected scores after 84.7s
Scanned 100,000,000 S2F rows; found 0/1,593 selected scores after 106.0s
Scanned 120,000,000 S2F rows; found 0/1,593 selected scores after 127.0s
Scanned 140,000,000 S2F rows; found 0/1,593 selected scores after 147.0s
Scanned 160,000,000 S2F rows; found 0/1,593 selected scores after 166.8s
Scanned 180,000,000 S2F rows; found 0/1,593 selected scores after 186.9s
Scanned 200,000,000 S2F rows; found 0/1,593 selected scores after 210.4s
Scanned 220,000,000 S2F rows; found 0/1,593 selected scores after 233.0s
Scanned 240,000,000 S2F rows; found 0/1,593 selected scores after 255.7s
Scanned

,dataset_id,organism,organism_key,domain,domain_name,available,prediction_path,protein_count,accuracy,macro_f1,balanced_accuracy
0,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,True,/media/marcelo_baez/HD_Disc1/.S2F/output/83333...,60.0,0.783333,0.783171,0.783333
1,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,True,/media/marcelo_baez/HD_Disc1/.S2F/output/83333...,60.0,0.600000,0.568307,0.600000
2,ecoli_cc,Escherichia coli,ecoli,CC,Cellular Component,True,/media/marcelo_baez/HD_Disc1/.S2F/output/83333...,60.0,0.883333,0.879749,0.883333
3,human_mf,Homo sapiens,human,MF,Molecular Function,False,NaN,NaN,NaN,NaN,NaN
4,human_bp,Homo sapiens,human,BP,Biological Process,False,NaN,NaN,NaN,NaN,NaN
5,human_cc,Homo sapiens,human,CC,Cellular Component,False,NaN,NaN,NaN,NaN,NaN


## Export Frontend Data


In [7]:
def finite_or_none(value):
    if value is None:
        return None
    try:
        if not math.isfinite(float(value)):
            return None
        return float(value)
    except Exception:
        return value


def dataframe_records(df):
    return json.loads(df.replace({np.nan: None}).to_json(orient='records'))

knn_by_dataset = {}
for dataset_id, dataset in datasets.items():
    summary = KNN_SUMMARY_DF[KNN_SUMMARY_DF['dataset_id'] == dataset_id].copy()
    predictions = KNN_PREDICTION_DF[KNN_PREDICTION_DF['dataset_id'] == dataset_id].copy()
    knn_by_dataset[dataset_id] = {
        'metadata': {
            'dataset_id': dataset_id,
            'organism': dataset['metadata']['organism'],
            'organism_key': dataset['metadata']['organism_key'],
            'domain': dataset['metadata']['domain'],
            'domain_name': dataset['metadata']['domain_name'],
            'method': 'leave-one-out weighted kNN over ESM cosine similarity',
            'sequence_identity': 'BLASTP coverage-adjusted pident: pident * alignment_length / max(query_length, subject_length)',
        },
        'identity_filters': IDENTITY_FILTERS,
        'k_values': K_VALUES,
        'summary': dataframe_records(summary),
        'predictions': dataframe_records(predictions),
        'sequence_identity_matrix': np.round(sequence_identity_matrices[dataset_id], 4).tolist(),
    }

linear_by_dataset = {}
summary_by_id = {row['dataset_id']: row for row in linear_summaries}
for dataset_id, dataset in datasets.items():
    predictions = LINEAR_PREDICTION_DF[LINEAR_PREDICTION_DF['dataset_id'] == dataset_id].copy()
    summary = summary_by_id[dataset_id]
    linear_by_dataset[dataset_id] = {
        'metadata': {
            'dataset_id': dataset_id,
            'organism': dataset['metadata']['organism'],
            'organism_key': dataset['metadata']['organism_key'],
            'domain': dataset['metadata']['domain'],
            'domain_name': dataset['metadata']['domain_name'],
            'method': summary['model'],
            'cv': summary['cv'],
        },
        'summary': summary,
        'predictions': dataframe_records(predictions),
    }

s2f_by_dataset = {}
s2f_summary_by_id = {row['dataset_id']: row for row in s2f_summaries}
for dataset_id, dataset in datasets.items():
    predictions = [row for row in s2f_predictions if row['dataset_id'] == dataset_id]
    s2f_by_dataset[dataset_id] = {
        'metadata': {
            'dataset_id': dataset_id,
            'organism': dataset['metadata']['organism'],
            'organism_key': dataset['metadata']['organism_key'],
            'domain': dataset['metadata']['domain'],
            'domain_name': dataset['metadata']['domain_name'],
        },
        'summary': s2f_summary_by_id[dataset_id],
        'predictions': predictions,
    }

comparison_rows = []
for dataset_id, dataset in datasets.items():
    knn_default = KNN_SUMMARY_DF[
        (KNN_SUMMARY_DF['dataset_id'] == dataset_id)
        & (KNN_SUMMARY_DF['identity_filter'] == 'all')
        & (KNN_SUMMARY_DF['k'] == 5)
    ].iloc[0]
    knn_low_identity = KNN_SUMMARY_DF[
        (KNN_SUMMARY_DF['dataset_id'] == dataset_id)
        & (KNN_SUMMARY_DF['identity_filter'] == 'lt30')
        & (KNN_SUMMARY_DF['k'] == 5)
    ].iloc[0]
    linear_summary = summary_by_id[dataset_id]
    method_rows = [
        ('ESM kNN k=5', 'all_neighbors', knn_default),
        ('ESM kNN k=5 <30% identity', 'sequence_filtered', knn_low_identity),
        ('Linear probe LOOCV', 'linear_probe', linear_summary),
    ]
    s2f_summary = s2f_summary_by_id[dataset_id]
    if s2f_summary.get('available'):
        method_rows.append(('S2F selected-term score', 's2f', s2f_summary))
    for method_label, method_key, metrics in method_rows:
        comparison_rows.append({
            'dataset_id': dataset_id,
            'organism': dataset['metadata']['organism'],
            'organism_key': dataset['metadata']['organism_key'],
            'domain': dataset['metadata']['domain'],
            'domain_name': dataset['metadata']['domain_name'],
            'method': method_label,
            'method_key': method_key,
            'accuracy': finite_or_none(metrics.get('accuracy')),
            'macro_f1': finite_or_none(metrics.get('macro_f1')),
            'balanced_accuracy': finite_or_none(metrics.get('balanced_accuracy')),
            'coverage': finite_or_none(metrics.get('coverage', 1.0)),
        })

COMPARISON_DF = pd.DataFrame(comparison_rows)

probe_payload = {
    'schema_version': 1,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': {
        'embedding_model': 'facebook/esm1b_t33_650M_UR50S',
        'base_frontend_index': 'data/index.json',
        'datasets': list(datasets.keys()),
    },
    'knn': {
        'description': 'Leave-one-out GO-group transfer from nearest ESM cosine-similarity neighbors, with optional BLASTP sequence-identity filtering.',
        'datasets': knn_by_dataset,
    },
    'linear_probe': {
        'description': 'Leave-one-out linear logistic-regression probe over frozen ESM embeddings.',
        'datasets': linear_by_dataset,
    },
    's2f': {
        'description': 'Selected GO-term ranking from local S2F prediction scores when available.',
        'datasets': s2f_by_dataset,
    },
    'comparison': {
        'rows': dataframe_records(COMPARISON_DF),
    },
}

with PROBE_RESULTS_PATH.open('w', encoding='utf-8') as handle:
    json.dump(probe_payload, handle, separators=(',', ':'), allow_nan=False)
KNN_SUMMARY_DF.to_csv(KNN_SUMMARY_PATH, index=False)
LINEAR_SUMMARY_DF.to_csv(LINEAR_SUMMARY_PATH, index=False)
COMPARISON_DF.to_csv(S2F_COMPARISON_PATH, index=False)

index_payload['prediction_probe_path'] = 'data/prediction_probe_results.json'
index_payload['prediction_probe_schema_version'] = 1
with INDEX_PATH.open('w', encoding='utf-8') as handle:
    json.dump(index_payload, handle, indent=2, sort_keys=True, allow_nan=False)

print('Wrote', PROBE_RESULTS_PATH)
print('Wrote', KNN_SUMMARY_PATH)
print('Wrote', LINEAR_SUMMARY_PATH)
print('Wrote', S2F_COMPARISON_PATH)
display(COMPARISON_DF)


Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/prediction_probe_results.json
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/knn_transfer_summary.csv
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/linear_probe_summary.csv
Wrote /home/marcelo_baez/paccanaro-lab/S2F/notebooks/esm_go_explorer/data/s2f_probe_comparison.csv


,dataset_id,organism,organism_key,domain,domain_name,method,method_key,accuracy,macro_f1,balanced_accuracy,coverage
0,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,ESM kNN k=5,all_neighbors,0.783333,0.781165,0.783333,1.0
1,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,ESM kNN k=5 <30% identity,sequence_filtered,0.766667,0.764926,0.766667,1.0
2,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,Linear probe LOOCV,linear_probe,0.850000,0.849817,0.850000,1.0
3,ecoli_mf,Escherichia coli,ecoli,MF,Molecular Function,S2F selected-term score,s2f,0.783333,0.783171,0.783333,1.0
4,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,ESM kNN k=5,all_neighbors,0.750000,0.739044,0.750000,1.0
5,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,ESM kNN k=5 <30% identity,sequence_filtered,0.750000,0.739044,0.750000,1.0
6,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,Linear probe LOOCV,linear_probe,0.816667,0.816667,0.816667,1.0
7,ecoli_bp,Escherichia coli,ecoli,BP,Biological Process,S2F selected-term score,s2f,0.600000,0.568307,0.600000,1.0
8,ecoli_cc,Escherichia coli,ecoli,CC,Cellular Component,ESM kNN k=5,all_neighbors,0.666667,0.675926,0.666667,1.0
9,ecoli_cc,Escherichia coli,ecoli,CC,Cellular Component,ESM kNN k=5 <30% identity,sequence_filtered,0.666667,0.675926,0.666667,1.0


## Interpretation Notes

- kNN transfer is still an embedding-space probe, not a trained PFP model.
- The sequence-identity filters remove candidate neighbors above the threshold; low coverage at strict filters means the selected 60-protein set has too few distant neighbors for that setting.
- The linear probe is intentionally simple. Strong performance means the selected labels are linearly accessible from frozen ESM embeddings.
- The S2F comparison is only shown when local prediction scores are available for the organism. In this run, E. coli scores are available; human scores are not configured locally.
